In [1]:
from gliclass.data_processing import GLiClassDataset
from transformers import AutoModel, AutoConfig
from gliclass.config import GLiClassModelConfig
from gliclass.model import GLiClassModel, GLiClassBiEncoder, GLiClassAudio
from transformers import AutoConfig, AutoTokenizer, AutoFeatureExtractor, ClapModel
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
import torchaudio, torch

/home/werent4/GLiClass/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/werent4/GLiClass/.venv/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
encoder_config = AutoConfig.from_pretrained("answerdotai/ModernBERT-base")
audiocfg = AutoConfig.from_pretrained("laion/clap-htsat-fused")

In [3]:
audio_encoder = ClapModel.from_pretrained("laion/clap-htsat-fused").audio_model
audio_feature_extractor = AutoFeatureExtractor.from_pretrained("laion/clap-htsat-fused")

In [4]:
glicalss_config = GLiClassModelConfig(
    encoder_config=encoder_config,
    encoder_model="answerdotai/ModernBERT-base",
    audio_model_name="laion/clap-htsat-fused",
    audio_model_config=audiocfg,
    class_token_index=len(tokenizer),
    text_token_index=len(tokenizer)+1,
    audio_token_index=len(tokenizer)+2, 
    pooling_strategy="first",
    scorer_type="simple",
    use_lstm=False,
    focal_loss_alpha=-1,
    focal_loss_gamma=-1,
    contrastive_loss_coef=0.0,
    normalize_features=False,
    extract_text_features=False,
    architecture_type='audio-encoder',
    prompt_first=True,
    squeeze_layers=False,
    shuffle_labels=True
)

model = GLiClassModel(glicalss_config, from_pretrained=True)
new_words = ["<<LABEL>>", "<<SEP>>", "<<AUDIO>>"]
tokenizer.add_tokens(new_words, special_tokens=True)
model.resize_token_embeddings(len(tokenizer), None)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50371, 768, padding_idx=50283)

In [5]:
import json
data = json.load(open("./datasets/processed_dataset.json", "r"))

In [6]:
train_dataset = GLiClassDataset(
    data,
    tokenizer,
    1024,
    "multi_label_classification",
    'audio-encoder',
    True,
    labels_tokenizer=tokenizer,
    audio_features_extractor= audio_feature_extractor,
    sampling_rate= 48000,
    max_duration_s=20
)

Total labels:  7
Audio parameters: sampling_rate=48000, max_duration=20s, max_samples=960000


In [7]:
exmpl = train_dataset[0]

In [8]:
input_ids = torch.tensor(exmpl['input_ids']).unsqueeze(0)  
attention_mask = torch.tensor(exmpl['attention_mask']).unsqueeze(0)  
labels = torch.tensor(exmpl['labels']).unsqueeze(0)  

/var/tmp/ipykernel_1307828/2331616996.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = torch.tensor(exmpl['input_ids']).unsqueeze(0)
/var/tmp/ipykernel_1307828/2331616996.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask = torch.tensor(exmpl['attention_mask']).unsqueeze(0)
/var/tmp/ipykernel_1307828/2331616996.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(exmpl['labels']).unsqueeze(0)


In [9]:
len(exmpl['input_ids'][0])

26

In [10]:
tokenizer.decode(exmpl['input_ids'][0].tolist())

'[CLS]<<LABEL>>Suprised<<LABEL>>Disgusted<<LABEL>>Sad<<LABEL>>Happy<<LABEL>>Neutral<<LABEL>>Fearful<<LABEL>>Angry<<SEP>><<AUDIO>>[SEP]'

In [11]:
exmpl["input_audio_features"].shape

torch.Size([1, 4, 1001, 64])

In [12]:
model.config.audio_token_index

50370

In [15]:
model(input_ids.squeeze(0), attention_mask.squeeze(0), exmpl["input_audio_features"], exmpl["is_longer"] ,labels=labels, return_dict=True)

audio_embeddings.shape:  torch.Size([1, 1, 768])
torch.Size([1, 26, 768])
inputs_embeds.shape:  torch.Size([1, 26, 768])
input_ids.shape:  torch.Size([1, 26])
attention_mask.shape:  torch.Size([1, 26])
audio_embeddings.shape:  torch.Size([1, 1, 768])
audio_seq_len:  1
before_audio.shape,  torch.Size([24, 768])
after_audio.shape,  torch.Size([1, 768])
audio_embeddings[batch_idx].shape,  torch.Size([1, 768])


GLiClassOutput(loss=tensor(14.2312, grad_fn=<NegBackward0>), logits=tensor([[-21.8982,  -9.3682, -11.0145, -19.9805, -31.6756, -14.8435,   4.8630]],
       grad_fn=<ViewBackward0>), hidden_states=None, attentions=None, text_embeddings=tensor([[ 2.5109e-01,  0.0000e+00,  7.1699e-01, -8.1399e-02,  1.3938e+00,
         -2.1293e+00, -0.0000e+00,  1.6679e+00,  1.0843e-01,  8.4658e-01,
         -7.1842e-01, -0.0000e+00,  0.0000e+00, -1.0825e+00,  1.6641e-01,
          8.8837e-01,  1.5308e+00,  1.7205e+00,  7.6500e-01, -8.8520e-01,
         -2.2832e+00,  0.0000e+00,  0.0000e+00,  2.5846e-01,  1.1437e+00,
         -1.8433e-01, -1.6722e-01, -1.2930e+00,  0.0000e+00,  1.3488e-01,
          0.0000e+00, -4.6935e-01,  8.7729e-01,  8.0536e-01, -3.6270e-01,
          3.9179e-01,  0.0000e+00,  3.9012e-01,  5.6695e-01,  8.6755e-01,
         -5.5539e-01,  6.3083e-01, -8.9871e-01,  1.6759e+00, -6.9187e-01,
          1.2598e-01,  1.7953e+00, -6.9820e-01,  1.1156e-01,  5.8671e-01,
         -1.2892e+00, -7.

In [ ]:
print("input_ids type:", type(exmpl['input_ids']))
print("attention_mask type:", type(exmpl['attention_mask']))
print("labels type:", type(exmpl['labels']))

input_ids type: <class 'list'>
attention_mask type: <class 'list'>
labels type: <class 'torch.Tensor'>


In [ ]:
input_ids = torch.tensor(exmpl['input_ids']).unsqueeze(0)  
attention_mask = torch.tensor(exmpl['attention_mask']).unsqueeze(0)  
labels = torch.tensor(exmpl['labels']).unsqueeze(0)  

/var/tmp/ipykernel_29360/2331616996.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(exmpl['labels']).unsqueeze(0)


In [ ]:
exmpl['labels']

tensor([1., 0., 0., 0., 0., 0., 0.])

In [11]:
print("Labels shape:", labels.shape)
print("Labels:", labels)
print("Problem type:", model.config.problem_type)

Labels shape: torch.Size([1, 7])
Labels: tensor([[1., 0., 0., 0., 0., 0., 0.]])
Problem type: None


In [12]:
exmpl['audio_input'].unsqueeze(0).shape

torch.Size([1, 80000])

In [14]:
model(input_ids, attention_mask, exmpl["audio_input"], labels=labels, return_dict=True)

GLiClassOutput(loss=tensor(4.1938, grad_fn=<NegBackward0>), logits=tensor([[-0.5545,  3.1775,  0.8603,  0.8491,  1.9720, -0.0429, -0.4186]],
       grad_fn=<MulBackward0>), hidden_states=None, attentions=None, text_embeddings=None, class_embeddings=None)

In [18]:
import torch, torchaudio

audio_encoder = ClapModel.from_pretrained("laion/clap-htsat-fused").audio_model
audio_feature_extractor = AutoFeatureExtractor.from_pretrained("laion/clap-htsat-fused")

In [17]:
len([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0,])

28